In [3]:
from pathlib import Path
import os

# Cari root project
PROJECT_ROOT = Path.cwd()

# Kalau notebook dijalankan dari folder notebooks/, naik ke folder project
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
    os.chdir(PROJECT_ROOT)

# Buat folder yang dibutuhkan
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PROMPTS_DIR = PROJECT_ROOT / "prompts"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

# Pakai absolute path untuk database
DB_PATH = DATA_DIR / "company_bi.db"

print("Project root:", PROJECT_ROOT)
print("Database path:", DB_PATH)
print("Folder data ada:", DATA_DIR.exists())
print("Database sudah ada:", DB_PATH.exists())

Project root: /Users/user/Library/CloudStorage/OneDrive-Personal/Documents/Work/Talentaa/AI Engineer/PT Seger Agro Nusantara/praktik/ai-bi-session-2
Database path: /Users/user/Library/CloudStorage/OneDrive-Personal/Documents/Work/Talentaa/AI Engineer/PT Seger Agro Nusantara/praktik/ai-bi-session-2/data/company_bi.db
Folder data ada: True
Database sudah ada: True


In [4]:
import sqlite3
import pandas as pd
from pathlib import Path

Path("data").mkdir(exist_ok=True)

con = sqlite3.connect(DB_PATH)
sales = pd.DataFrame([
    {"date": "2026-03-01", "channel": "TikTok", "sku": "SSUP", "qty": 30, "revenue": 6720000, "region": "Jawa Barat"},
    {"date": "2026-03-03", "channel": "Shopee", "sku": "ADC", "qty": 24, "revenue": 5376000, "region": "DKI Jakarta"},
    {"date": "2026-03-05", "channel": "Tokopedia", "sku": "TONER", "qty": 18, "revenue": 3600000, "region": "Jawa Tengah"},

    {"date": "2026-04-01", "channel": "TikTok", "sku": "SSUP", "qty": 38, "revenue": 8512000, "region": "Jawa Barat"},
    {"date": "2026-04-03", "channel": "Shopee", "sku": "ADC", "qty": 28, "revenue": 6272000, "region": "DKI Jakarta"},
    {"date": "2026-04-05", "channel": "Tokopedia", "sku": "TONER", "qty": 22, "revenue": 4400000, "region": "Jawa Tengah"},

    {"date": "2026-05-01", "channel": "TikTok", "sku": "SSUP", "qty": 45, "revenue": 10080000, "region": "Jawa Barat"},
    {"date": "2026-05-02", "channel": "TikTok", "sku": "ADC", "qty": 20, "revenue": 4480000, "region": "Jawa Timur"},
    {"date": "2026-05-03", "channel": "Shopee", "sku": "ADC", "qty": 30, "revenue": 6720000, "region": "DKI Jakarta"},
    {"date": "2026-05-04", "channel": "Shopee", "sku": "TONER", "qty": 16, "revenue": 3200000, "region": "Banten"},
    {"date": "2026-05-05", "channel": "Tokopedia", "sku": "SSUP", "qty": 25, "revenue": 5600000, "region": "Jawa Tengah"},
    {"date": "2026-05-06", "channel": "Website", "sku": "BUNDLE", "qty": 12, "revenue": 7200000, "region": "DKI Jakarta"},
])

hr = pd.DataFrame([
    {"month": "2026-03", "department": "Sales", "employee_count": 33, "attrition": 1},
    {"month": "2026-03", "department": "Ops", "employee_count": 21, "attrition": 1},
    {"month": "2026-03", "department": "Finance", "employee_count": 10, "attrition": 0},

    {"month": "2026-04", "department": "Sales", "employee_count": 34, "attrition": 2},
    {"month": "2026-04", "department": "Ops", "employee_count": 22, "attrition": 1},
    {"month": "2026-04", "department": "Finance", "employee_count": 10, "attrition": 0},

    {"month": "2026-05", "department": "Sales", "employee_count": 35, "attrition": 2},
    {"month": "2026-05", "department": "Ops", "employee_count": 22, "attrition": 1},
    {"month": "2026-05", "department": "Finance", "employee_count": 11, "attrition": 0},
])

finance = pd.DataFrame([
    {"month": "2026-03", "cost_center": "Marketing", "expense": 38000000},
    {"month": "2026-03", "cost_center": "Fulfillment", "expense": 19000000},
    {"month": "2026-03", "cost_center": "Customer Support", "expense": 9000000},

    {"month": "2026-04", "cost_center": "Marketing", "expense": 42000000},
    {"month": "2026-04", "cost_center": "Fulfillment", "expense": 20500000},
    {"month": "2026-04", "cost_center": "Customer Support", "expense": 9500000},

    {"month": "2026-05", "cost_center": "Marketing", "expense": 45000000},
    {"month": "2026-05", "cost_center": "Fulfillment", "expense": 22000000},
    {"month": "2026-05", "cost_center": "Customer Support", "expense": 10000000},
])

sales.to_sql("sales_orders", con, if_exists="replace", index=False)
hr.to_sql("hr_monthly", con, if_exists="replace", index=False)
finance.to_sql("finance_expense", con, if_exists="replace", index=False)

con.close()

print("Database sample berhasil dibuat")


Database sample berhasil dibuat


In [60]:
print("Database berhasil dibuat di folder data/company_bi.db")

Database berhasil dibuat di folder data/company_bi.db


In [5]:
def get_schema_summary(con):
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table'",
        con
    )["name"].tolist()

    lines = []
    for table in tables:
        cols = pd.read_sql_query(f"PRAGMA table_info({table})", con)
        lines.append(f"Table: {table}")
        for _, row in cols.iterrows():
            lines.append(f"- {row['name']} {row['type']}")
        lines.append("")

    business_notes = """
Business definitions:
- revenue = nilai penjualan kotor dari sales_orders.
- qty = jumlah unit produk terjual.
- attrition = jumlah karyawan keluar pada periode tertentu.
- expense = biaya aktual pada cost center.
Rules:
- Gunakan hanya tabel yang tersedia di schema.
- Jangan gunakan kolom yang tidak muncul di schema.
"""
    return "\n".join(lines) + business_notes

con = sqlite3.connect(DB_PATH)
schema_summary = get_schema_summary(con)
con.close()
print(schema_summary)


Table: sales_orders
- date TEXT
- channel TEXT
- sku TEXT
- qty INTEGER
- revenue INTEGER
- region TEXT

Table: hr_monthly
- month TEXT
- department TEXT
- employee_count INTEGER
- attrition INTEGER

Table: finance_expense
- month TEXT
- cost_center TEXT
- expense INTEGER

Business definitions:
- revenue = nilai penjualan kotor dari sales_orders.
- qty = jumlah unit produk terjual.
- attrition = jumlah karyawan keluar pada periode tertentu.
- expense = biaya aktual pada cost center.
Rules:
- Gunakan hanya tabel yang tersedia di schema.
- Jangan gunakan kolom yang tidak muncul di schema.



In [6]:
def build_sql_prompt(question, schema):
    return f"""
Anda adalah data analyst yang menulis SQL SQLite.
Tugas: buat SQL untuk menjawab pertanyaan user.

Rules wajib:
1. Hanya SELECT. Tidak boleh INSERT/UPDATE/DELETE/DROP.
2. Gunakan hanya tabel dan kolom dari schema.
3. Selalu tambahkan LIMIT 100.
4. Jika pertanyaan ambigu, set clarification_needed = true.
5. Jika data yang diminta tidak tersedia, jelaskan lewat clarification_question.

Schema:
{schema}

Question:
{question}

Return JSON only with keys:
- sql
- assumptions
- clarification_needed
- clarification_question
"""


In [7]:
from openai import OpenAI
import json
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [8]:
from openai import OpenAI
import json
import os

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")
)

def clean_json_output(text):
    """
    Membersihkan output model jika JSON dibungkus markdown seperti:
    ```json
    {...}
    ```
    """
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "", 1).strip()

    if text.startswith("```"):
        text = text.replace("```", "", 1).strip()

    if text.endswith("```"):
        text = text[:-3].strip()

    return text


def generate_sql(question, schema):
    prompt = build_sql_prompt(question, schema)

    res = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    output_text = clean_json_output(res.output_text)

    try:
        return json.loads(output_text)
    except json.JSONDecodeError:
        raise ValueError(f"Output model bukan JSON valid:\n{res.output_text}")


result = generate_sql(
    "Berapa revenue per channel bulan Mei?",
    schema_summary
)

print(result["sql"])

SELECT channel, SUM(revenue) AS total_revenue FROM sales_orders WHERE date LIKE '2023-05%' GROUP BY channel LIMIT 100;


In [9]:
import re
import sqlglot
from sqlglot import expressions as exp

FORBIDDEN_PATTERN = r"\b(insert|update|delete|drop|alter|create|replace|attach|detach|pragma)\b"

ALLOWED_TABLES = {"sales_orders", "hr_monthly", "finance_expense"}

ALLOWED_COLUMNS = {
    "sales_orders": {"date", "channel", "sku", "qty", "revenue", "region"},
    "hr_monthly": {"month", "department", "employee_count", "attrition"},
    "finance_expense": {"month", "cost_center", "expense"},
}

def validate_sql(sql):
    cleaned = sql.strip()
    low = cleaned.lower()

    # Hanya boleh satu statement
    if ";" in low.rstrip(";"):
        raise ValueError("Multiple statements are not allowed")

    # Wajib SELECT
    parsed = sqlglot.parse_one(cleaned, read="sqlite")
    if parsed.key != "select":
        raise ValueError("Query must be SELECT")

    # Blok keyword berbahaya
    if re.search(FORBIDDEN_PATTERN, low):
        raise ValueError("Only safe SELECT query is allowed")

    # Validasi tabel
    tables = {t.name for t in parsed.find_all(exp.Table)}
    if not tables <= ALLOWED_TABLES:
        raise ValueError(f"Table not allowed: {tables - ALLOWED_TABLES}")

    # Validasi kolom
    columns = {c.name for c in parsed.find_all(exp.Column)}
    allowed_cols = set()
    for table in tables:
        allowed_cols.update(ALLOWED_COLUMNS[table])

    if not columns <= allowed_cols:
        raise ValueError(f"Column not allowed: {columns - allowed_cols}")

    # Wajib LIMIT maksimal 100
    limit_match = re.search(r"\blimit\s+(\d+)\b", low)
    if not limit_match:
        raise ValueError("LIMIT is required")

    limit_value = int(limit_match.group(1))
    if limit_value > 100:
        raise ValueError("LIMIT must be 100 or less")

    return True


In [10]:
import pandas as pd
import sqlite3
from pathlib import Path

def execute_sql(sql, db_path=DB_PATH):
    validate_sql(sql)

    db_path = Path(db_path).resolve()

    print("Database yang dipakai execute_sql:", db_path)

    if not db_path.exists():
        raise FileNotFoundError(f"Database tidak ditemukan: {db_path}")

    # Buka database dalam mode read-only
    db_uri = f"file:{db_path}?mode=ro"
    con = sqlite3.connect(db_uri, uri=True, timeout=5)

    try:
        df = pd.read_sql_query(sql, con)
    finally:
        con.close()

    if len(df) > 100:
        df = df.head(100)

    return df


df_result = execute_sql(result["sql"])
df_result

Database yang dipakai execute_sql: /Users/user/Library/CloudStorage/OneDrive-Personal/Documents/Work/Talentaa/AI Engineer/PT Seger Agro Nusantara/praktik/ai-bi-session-2/data/company_bi.db


,channel,total_revenue


In [11]:
def synthesize_answer(question, sql, df):
    if df.empty:
        table = "Tidak ada data yang cocok dengan query."
    else:
        table = df.to_markdown(index=False)

    prompt = f"""
Anda adalah BI analyst. Jawab pertanyaan user berdasarkan hasil query berikut.

Question:
{question}

SQL:
{sql}

Result:
{table}

Format jawaban:
1. Jawaban langsung dalam 1 paragraf
2. 3 insight utama
3. Asumsi atau keterbatasan data

Aturan:
- Jangan mengarang angka di luar result.
- Jika data kosong, jelaskan bahwa data tidak tersedia untuk filter tersebut.
- Gunakan bahasa Indonesia yang ringkas dan bisnis-friendly.
"""

    res = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    return res.output_text


In [12]:
def repair_sql(question, schema, previous_sql, error_message):
    prompt = f"""
Anda adalah SQL repair assistant untuk SQLite.

Tugas:
Perbaiki SQL yang gagal berdasarkan error berikut.

Question:
{question}

Schema:
{schema}

SQL sebelumnya:
{previous_sql}

Error:
{error_message}

Rules wajib:
1. Hanya SELECT.
2. Gunakan hanya tabel dan kolom dari schema.
3. Selalu tambahkan LIMIT 100.
4. Jika pertanyaan ambigu, set clarification_needed = true.
5. Return JSON only dengan keys:
   sql, assumptions, clarification_needed, clarification_question.
"""

    res = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    output_text = clean_json_output(res.output_text)

    try:
        return json.loads(output_text)
    except json.JSONDecodeError:
        raise ValueError(f"Output repair bukan JSON valid:\n{res.output_text}")


def ask_database(question, max_retry=2):
    con = sqlite3.connect("data/company_bi.db")
    auto_schema = get_schema_summary(con)
    con.close()

    # get_schema_summary already includes business notes,
    # avoid referencing undefined 'business_notes' variable
    schema = auto_schema

    last_error = None
    sql_payload = generate_sql(question, schema)

    for attempt in range(max_retry + 1):
        try:
            if sql_payload.get("clarification_needed"):
                return sql_payload["clarification_question"]

            sql = sql_payload["sql"]
            df = execute_sql(sql)

            return synthesize_answer(question, sql, df)

        except Exception as e:
            last_error = str(e)

            if attempt == max_retry:
                break

            sql_payload = repair_sql(
                question=question,
                schema=schema,
                previous_sql=sql_payload.get("sql", ""),
                error_message=last_error
            )

    return f"Query belum berhasil setelah retry. Error terakhir: {last_error}"


In [13]:
answer = ask_database("Berapa revenue per channel bulan Mei?")
print(answer)

questions = [
    "SKU paling laku di TikTok?",
    "Region mana yang omzetnya paling tinggi?",
    "Departemen mana dengan attrition tertinggi?",
    "Biaya terbesar bulan Mei di cost center apa?",
]

for q in questions:
    print("=" * 80)
    print("Question:", q)
    print(ask_database(q))


Database yang dipakai execute_sql: /Users/user/Library/CloudStorage/OneDrive-Personal/Documents/Work/Talentaa/AI Engineer/PT Seger Agro Nusantara/praktik/ai-bi-session-2/data/company_bi.db
Revenue per channel bulan Mei menunjukkan bahwa TikTok menjadi kanal dengan pendapatan tertinggi sebesar 14.560.000, diikuti oleh Shopee dengan 9.920.000, Website 7.200.000, dan Tokopedia 5.600.000. Tiga insight utama adalah: 1) TikTok unggul sebagai channel penjualan utama, menandakan efektivitas pemasaran atau cakupan pasar yang luas di platform tersebut; 2) Shopee dan Website juga memberikan kontribusi signifikan, sehingga perlu strategi peningkatan lebih lanjut untuk mempertahankan dan meningkatkan performa; 3) Tokopedia memiliki revenue paling rendah, yang bisa menjadi fokus untuk optimasi baik dari segi promosi maupun user experience. Asumsi yang perlu diperhatikan adalah data hanya mencakup bulan Mei tanpa informasi trend atau perbandingan bulan lain, serta tidak tersedia rincian jumlah transa

In [70]:
answer = ask_database("Total expense Marketing berapa?")
print(answer)

questions = [
    "ROI campaign berapa?",
    "Cost center mana yang naik?",
]

for q in questions:
    print("=" * 80)
    print("Question:", q)
    print(ask_database(q))


Database yang dipakai execute_sql: /Users/user/Library/CloudStorage/OneDrive-Personal/Documents/Work/Talentaa/AI Engineer/PT Seger Agro Nusantara/praktik/ai-bi-session-2/data/company_bi.db
Total expense untuk cost center Marketing adalah sebesar 125.000.000. 

Insight utama: 
1) Pengeluaran Marketing cukup signifikan, menunjukkan alokasi anggaran yang besar untuk aktivitas pemasaran. 
2) Total expense ini bisa menjadi acuan untuk evaluasi efektivitas pemasaran terhadap hasil bisnis. 
3) Dengan total expense yang tinggi, perlu dipantau secara berkala agar anggaran marketing digunakan dengan efisien.

Asumsi atau keterbatasan data: Data hanya mencakup expense dengan cost_center 'Marketing' dan tidak dijelaskan periode waktu sehingga analisa tren tidak dapat dilakukan. Data juga tidak merinci kategori expense dalam marketing yang dapat membantu analisis lebih detail.
Question: ROI campaign berapa?
ROI campaign tidak dapat dihitung langsung karena tidak ada informasi tentang biaya kampanye

In [14]:
from datetime import datetime
from pathlib import Path
import time

Path("outputs").mkdir(exist_ok=True)

def log_query(question, sql, status, row_count=0, elapsed_ms=None, error=None):
    log_path = "outputs/query_log.csv"

    row = pd.DataFrame([{
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "question": question,
        "sql": sql,
        "status": status,
        "row_count": row_count,
        "elapsed_ms": elapsed_ms,
        "error": error
    }])

    file_exists = Path(log_path).exists()

    row.to_csv(
        log_path,
        mode="a",
        header=not file_exists,
        index=False
    )


In [15]:
def is_sensitive_hr_question(question):
    sensitive_terms = {
        "gaji",
        "salary",
        "upah",
        "nik",
        "no ktp",
        "nomor ktp",
        "alamat",
        "email",
        "telepon",
        "phone",
        "personal",
        "pribadi",
    }
    low = question.lower()
    return any(term in low for term in sensitive_terms)


def ask_database_logged(question, max_retry=2):
    if is_sensitive_hr_question(question):
        return (
            "Pertanyaan ini berpotensi mengakses data personal karyawan. "
            "Untuk praktik ini, sistem hanya menjawab data HR dalam bentuk agregat."
        )

    con = sqlite3.connect("data/company_bi.db")
    auto_schema = get_schema_summary(con)
    con.close()

    schema = auto_schema

    last_error = None
    last_sql = ""

    start_time = time.perf_counter()

    try:
        sql_payload = generate_sql(question, schema)

        for attempt in range(max_retry + 1):
            try:
                if sql_payload.get("clarification_needed"):
                    return sql_payload["clarification_question"]

                sql = sql_payload["sql"]
                last_sql = sql

                df = execute_sql(sql)

                elapsed_ms = round((time.perf_counter() - start_time) * 1000, 2)

                log_query(
                    question=question,
                    sql=sql,
                    status="success",
                    row_count=len(df),
                    elapsed_ms=elapsed_ms,
                    error=None
                )

                return synthesize_answer(question, sql, df)

            except Exception as e:
                last_error = str(e)

                if attempt == max_retry:
                    break

                sql_payload = repair_sql(
                    question=question,
                    schema=schema,
                    previous_sql=sql_payload.get("sql", ""),
                    error_message=last_error
                )

        elapsed_ms = round((time.perf_counter() - start_time) * 1000, 2)

        log_query(
            question=question,
            sql=last_sql,
            status="failed",
            row_count=0,
            elapsed_ms=elapsed_ms,
            error=last_error
        )

        return f"Query belum berhasil setelah retry. Error terakhir: {last_error}"

    except Exception as e:
        elapsed_ms = round((time.perf_counter() - start_time) * 1000, 2)

        log_query(
            question=question,
            sql=last_sql,
            status="failed",
            row_count=0,
            elapsed_ms=elapsed_ms,
            error=str(e)
        )

        return f"Terjadi error: {e}"

In [17]:
answer = ask_database_logged("Berapa revenue per channel bulan Mei?")
print(answer)

log_path = Path("outputs/query_log.csv")
if log_path.exists():
    log_df = pd.read_csv(log_path)
    log_df.tail()
else:
    print("query_log.csv belum dibuat.")


Database yang dipakai execute_sql: /Users/user/Library/CloudStorage/OneDrive-Personal/Documents/Work/Talentaa/AI Engineer/PT Seger Agro Nusantara/praktik/ai-bi-session-2/data/company_bi.db
Revenue per channel bulan Mei adalah Shopee sebesar 9.920.000, TikTok sebesar 14.560.000, Tokopedia sebesar 5.600.000, dan Website sebesar 7.200.000. 

Tiga insight utama: 1) TikTok menghasilkan revenue tertinggi di bulan Mei, menunjukkan efektivitas channel ini dalam menjangkau pelanggan atau konversi penjualan. 2) Tokopedia memiliki revenue terendah, yang mungkin mengindikasikan perlunya strategi pemasaran atau promosi yang lebih agresif di platform tersebut. 3) Shopee dan Website memiliki kontribusi revenue yang cukup signifikan namun masih kalah dari TikTok, sehingga ada peluang untuk mengoptimalkan kedua channel ini lebih lanjut.

Asumsi dan keterbatasan data: Data hanya mencakup revenue bulan Mei tanpa informasi mengenai volume penjualan atau margin keuntungan, serta tidak mempertimbangkan fakt